# Introduction to BERT

This notebook will give an introduction to inner working on BERT

In [ ]:
!pip install -q bertviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.1/664.8 MB 71.0 MB/s eta 0:00:07

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
from tensorflow.keras.layers import Input, Dense,Flatten

In [ ]:
#Create an instance of BERT tokenizer and model
model_name='bert-base-uncased'
from transformers import BertTokenizer, TFBertModel,AutoTokenizer, AutoModel, utils
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertModel.from_pretrained("bert-base-uncased")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

BERT is pretrained on two tasks, Masked language Modeling and next sentence Prediction.

**During Masked Language Modeling** 15% of the words in the input is masked, run the entire sequence through a deep bidirectional Transformer encoder, and then predict only the masked words.

For example:

Input: the man went to the [MASK1] . he bought a [MASK2] of milk.
Labels: [MASK1] = store; [MASK2] = gallon

**During next sentence prediction**, Given sentences A and B, BERT is trained to predict if B is actual next sentence after A, or a random sentence in the corpus.

Sentence A: the man went to the store .
Sentence B: he bought a gallon of milk .
Label: IsNextSentence

Sentence A: the man went to the store .
Sentence B: penguins are flightless .
Label: NotNextSentence

In [ ]:
text = ["i went to state bank india for a loan application", "a man started aggriculture on river bank ", "a man started his loan and investment application in a bank"]
encoded_input = tokenizer(text,padding=True, truncation=True, max_length=128, return_tensors="tf")
output = model(encoded_input)

In [ ]:
encoded_input.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

# Special Tokens in BERT:
[CLS] : The start token of every document. This is 101

[SEP] : Placed between each sentence. This is 102

[PAD] : Added at the end of the document to keep the length of document to 512

'##token' : Indicates the start of a "word piece."

In [ ]:
encoded_input['input_ids']

<tf.Tensor: shape=(3, 13), dtype=int32, numpy=
array([[  101,  1045,  2253,  2000,  2110,  2924,  2634,  2005,  1037,
         5414,  4646,   102,     0],
       [  101,  1037,  2158,  2318, 12943, 16523,  2594, 11314,  5397,
         2006,  2314,  2924,   102],
       [  101,  1037,  2158,  2318,  2010,  5414,  1998,  5211,  4646,
         1999,  1037,  2924,   102]], dtype=int32)>

In [ ]:
encoded_input['token_type_ids']

<tf.Tensor: shape=(3, 13), dtype=int32, numpy=
array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=int32)>

In [ ]:
encoded_input['attention_mask']

<tf.Tensor: shape=(3, 13), dtype=int32, numpy=
array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], dtype=int32)>

Encode and Decode function of tokenizer can be used to probe and check the output

In [ ]:
tokenizer.encode('bank is holiday')

[101, 2924, 2003, 6209, 102]

In [ ]:
tokenizer.decode(2924)

'bank'

In [ ]:
output.keys()

odict_keys(['last_hidden_state', 'pooler_output'])

Pooler Output is the sentence encoding of BERT model. These are the pretrained vectors and can be used as input to further finetune specific NLP tasks.

In [ ]:
output['pooler_output'].shape

TensorShape([3, 768])

last_hidden_state is the output of the last state, these are the individual word embeddings

In [ ]:
output['last_hidden_state'].shape

TensorShape([3, 13, 768])

In [ ]:
output['last_hidden_state']

<tf.Tensor: shape=(2, 5, 768), dtype=float32, numpy=
array([[[-0.51829386, -0.4049461 , -0.5799782 , ..., -0.1981639 ,
          0.21114515,  0.5328875 ],
        [ 0.74305135, -0.3559298 , -0.95778465, ...,  0.21998604,
          0.39417338, -0.3687524 ],
        [ 0.09384765, -0.48176765, -1.4730046 , ..., -1.0073861 ,
         -0.18800849, -0.87084615],
        [-0.45732644, -0.3237682 , -1.2384443 , ...,  0.01464262,
         -0.37701407, -0.3317703 ],
        [ 0.83600885, -0.07839366, -0.33794415, ..., -0.10233694,
         -0.60716546, -0.24850394]],

       [[-0.1755962 ,  0.00390512, -0.3966071 , ..., -0.49171975,
          0.34756392,  0.464173  ],
        [ 0.62480587, -0.0410549 , -0.62988263, ..., -0.6588109 ,
         -0.61864674, -0.4013143 ],
        [ 0.41485724, -0.113434  , -0.43257713, ..., -0.7051331 ,
         -0.5263005 , -0.65657777],
        [ 0.2726929 ,  0.09262171, -0.62111664, ..., -0.41310725,
          0.06437566, -0.75378495],
        [ 0.6449392 ,  0.04

In [ ]:
bank1=output['last_hidden_state'][0][5]
bank2=output['last_hidden_state'][1][11]
bank3=output['last_hidden_state'][2][11]

In [ ]:
import numpy as np
from numpy.linalg import norm

In [ ]:
cosine1 = np.dot(bank1,bank2)/(norm(bank1)*norm(bank2))
print(cosine1)

cosine2 = np.dot(bank1,bank3)/(norm(bank1)*norm(bank3))
print(cosine2)

cosine3 = np.dot(bank2,bank3)/(norm(bank2)*norm(bank3))
print(cosine3)



0.431614
0.5550127
0.68497664


BERT in action.
bertviz can be used to visualize the workings of BERT.

https://github.com/jessevig/bertviz

In [ ]:

from transformers import AutoTokenizer, AutoModel, utils
from bertviz import model_view,head_view
utils.logging.set_verbosity_error()  # Suppress standard warnings

model_name = "google-bert/bert-base-uncased"  # Find popular HuggingFace models here: https://huggingface.co/models

model = AutoModel.from_pretrained(model_name, output_attentions=True)  # Configure model to return attention values
tokenizer = AutoTokenizer.from_pretrained(model_name)
sentence_a = "This is a Green house"
sentence_b = "The house is painted Green"
inputs = tokenizer.encode_plus(sentence_a, sentence_b, return_tensors='pt')
input_ids = inputs['input_ids']
token_type_ids = inputs['token_type_ids']
attention = model(input_ids, token_type_ids=token_type_ids)[-1]
sentence_b_start = token_type_ids[0].tolist().index(1)
input_id_list = input_ids[0].tolist() # Batch index 0
tokens = tokenizer.convert_ids_to_tokens(input_id_list)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
head_view(attention, tokens, sentence_b_start)

<IPython.core.display.Javascript object>

In [ ]:
model_view(attention, tokens, sentence_b_start)

<IPython.core.display.Javascript object>